# 07 — kvpress: RULER Benchmark with PrefillDecodingPress

This notebook evaluates KV cache compression on the
[RULER benchmark](https://huggingface.co/datasets/simonjegou/ruler) using
[kvpress](https://github.com/NVIDIA/kvpress) with Qwen3-8B.

We test **KeyDiffPress**-based compression applied to both **prefill** and
**decoding** phases using PrefillDecodingPress, with two decoding strategies:
- **full_replacement** — CompressionRatioDecodingPress
- **filtering** — FilteringPress

RULER consists of 13 synthetic long-context tasks (NIAH variants, variable
tracking, word extraction, QA) designed to stress-test retrieval and tracking
at controlled context lengths.

Scoring uses `calculate_metrics` from the kvpress evaluation framework
(same scoring as the [kvpress leaderboard](https://huggingface.co/spaces/nvidia/kvpress-leaderboard)).

Results are saved to `results/kvpress_ruler/` for comparison in later notebooks.

## Configuration

In [1]:
import os
os.environ["HF_HOME"] = "/opt/app-root/src/.cache/huggingface"

MODEL_NAME = "Qwen/Qwen3-8B"

COMPRESSION_RATIOS = [0.01, 0.25, 0.50, 0.75]

RULER_DATA_DIRS = ["4096", "8192"]

FRACTION = 0.01

SEED = 42

PRESS_CONFIGS = {
    "full_replacement": lambda cr: PrefillDecodingPress(
        prefilling_press=KeyDiffPress(compression_ratio=cr),
        decoding_press=CompressionRatioDecodingPress(
            base_press=KeyDiffPress(), target_compression_ratio=cr,
        ),
    ),
    "filtering": lambda cr: PrefillDecodingPress(
        prefilling_press=KeyDiffPress(compression_ratio=cr),
        decoding_press=FilteringPress(
            base_press=KeyDiffPress(), target_compression_ratio=cr,
            fill_padding=False,
        ),
    ),
}

In [2]:
import sys
import builtins

_original_print = builtins.print

def print(*args, **kwargs):
    _original_print(*args, **kwargs)
    if sys.stdout is not sys.__stdout__:
        kwargs['file'] = sys.__stdout__
        kwargs['flush'] = True
        _original_print(*args, **kwargs)

In [3]:
import sys
import os

FORK_DIR = "/opt/app-root/src/kvpress-fork"

if os.path.isdir(FORK_DIR) and os.listdir(FORK_DIR):
    sys.path.insert(0, FORK_DIR)
    import kvpress
    print(f"Using FORK kvpress from {FORK_DIR}")
else:
    import kvpress
    print(f"Using SYSTEM kvpress")

print(f"  location: {os.path.dirname(kvpress.__file__)}")

EVAL_DIR = os.path.join(FORK_DIR, "evaluation")
sys.path.insert(0, EVAL_DIR)
from benchmarks.ruler.calculate_metrics import calculate_metrics as ruler_calculate_metrics
print(f"  RULER scoring from: {EVAL_DIR}")

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using FORK kvpress from /opt/app-root/src/kvpress-fork
  location: /opt/app-root/src/kvpress-fork/kvpress
Using FORK kvpress from /opt/app-root/src/kvpress-fork
  location: /opt/app-root/src/kvpress-fork/kvpress
  RULER scoring from: /opt/app-root/src/kvpress-fork/evaluation
  RULER scoring from: /opt/app-root/src/kvpress-fork/evaluation


## 1. Load Model

In [4]:
import torch
from transformers import pipeline
from kvpress import (
    KeyDiffPress, PrefillDecodingPress, CompressionRatioDecodingPress,
    FilteringPress,
)

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)} — {vram_gb:.1f} GB VRAM")

model_kwargs = {}

try:
    import flash_attn  # noqa: F401
    model_kwargs["attn_implementation"] = "flash_attention_2"
    print("Using Flash Attention 2")
except ImportError:
    print("Flash Attention 2 not available, using default attention")

pipe = pipeline(
    "kv-press-text-generation",
    model=MODEL_NAME,
    device_map="auto",
    model_kwargs=model_kwargs,
    trust_remote_code=True,
)

print(f"\nModel loaded. GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU: NVIDIA A100-SXM4-40GB — 42.4 GB VRAM
Using Flash Attention 2
GPU: NVIDIA A100-SXM4-40GB — 42.4 GB VRAM
Using Flash Attention 2


Loading checkpoint shards: 100%|██████████| 5/5 [00:04<00:00,  1.01it/s]
Device set to use cuda:0



Model loaded. GPU memory allocated: 16.38 GB

Model loaded. GPU memory allocated: 16.38 GB


## 2. Load RULER Dataset

In [5]:
from datasets import load_dataset

ruler_datasets = {}
for data_dir in RULER_DATA_DIRS:
    df = load_dataset("simonjegou/ruler", data_dir=data_dir, split="test").to_pandas()
    if FRACTION < 1.0:
        df = df.sample(frac=FRACTION, random_state=SEED)
    ruler_datasets[data_dir] = df
    tasks = sorted(df["task"].unique())
    print(f"RULER {data_dir}: {len(df)} examples, {len(tasks)} tasks")
    print(f"  Tasks: {tasks}")

RULER 4096: 65 examples, 12 tasks
  Tasks: ['cwe', 'fwe', 'niah_multikey_1', 'niah_multikey_2', 'niah_multikey_3', 'niah_multiquery', 'niah_multivalue', 'niah_single_1', 'niah_single_2', 'niah_single_3', 'qa_1', 'qa_2']
RULER 4096: 65 examples, 12 tasks
  Tasks: ['cwe', 'fwe', 'niah_multikey_1', 'niah_multikey_2', 'niah_multikey_3', 'niah_multiquery', 'niah_multivalue', 'niah_single_1', 'niah_single_2', 'niah_single_3', 'qa_1', 'qa_2']
RULER 8192: 65 examples, 12 tasks
  Tasks: ['cwe', 'fwe', 'niah_multikey_1', 'niah_multikey_2', 'niah_multikey_3', 'niah_multiquery', 'niah_multivalue', 'niah_single_1', 'niah_single_2', 'niah_single_3', 'qa_1', 'qa_2']
RULER 8192: 65 examples, 12 tasks
  Tasks: ['cwe', 'fwe', 'niah_multikey_1', 'niah_multikey_2', 'niah_multikey_3', 'niah_multiquery', 'niah_multivalue', 'niah_single_1', 'niah_single_2', 'niah_single_3', 'qa_1', 'qa_2']


## 3. Run Inference

For each (algorithm, compression_ratio, context_length) combination, run all
RULER examples through the kvpress pipeline. Predictions are collected for
scoring in the next section using the kvpress evaluation framework.

In [6]:
import time
import random
import numpy as np
import pandas as pd

# Deterministic seeds — matches evaluate.py _setup_deterministic_seeds()
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

all_metrics = {}
summary_rows = []
all_predictions = []

configs = [("no_press", 0.0, None)]
for press_name, press_factory in PRESS_CONFIGS.items():
    for ratio in COMPRESSION_RATIOS:
        configs.append((press_name, ratio, press_factory(ratio)))

for press_name, ratio, press in configs:
    for data_dir in RULER_DATA_DIRS:
        df_eval = ruler_datasets[data_dir].copy()
        label = f"{press_name} | ratio={ratio} | ctx={data_dir}"
        print(f"\n{'='*60}")
        print(f"Running: {label} ({len(df_eval)} examples)")
        print(f"{'='*60}")

        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        log_every = max(1, len(df_eval) // 10)

        df_eval["predicted_answer"] = None
        elapsed_times = []

        for idx, (i, row) in enumerate(df_eval.iterrows()):
            kwargs = dict(
                question=row["question"],
                answer_prefix=row["answer_prefix"],
                max_new_tokens=row["max_new_tokens"],
            )
            if press is not None:
                kwargs["press"] = press

            t_start = time.perf_counter()
            output = pipe(row["context"], **kwargs)
            elapsed = time.perf_counter() - t_start

            df_eval.at[i, "predicted_answer"] = output["answer"]
            elapsed_times.append(elapsed)

            if (idx + 1) % log_every == 0 or (idx + 1) == len(df_eval):
                total_elapsed = time.perf_counter() - t0
                print(f"  {idx+1}/{len(df_eval)} — {total_elapsed:.0f}s elapsed")

        total_elapsed = time.perf_counter() - t0
        peak_mem = torch.cuda.max_memory_allocated() / 1e9
        print(f"  Done: {total_elapsed:.0f}s, peak_mem={peak_mem:.2f}GB")

        # Score using calculate_metrics — same flow as evaluate.py
        metrics = ruler_calculate_metrics(df_eval)
        key = f"{press_name}__{ratio}__{data_dir}"
        all_metrics[key] = metrics

        avg_score = sum(m["string_match"] for m in metrics.values()) / len(metrics)
        mean_time = sum(elapsed_times) / len(elapsed_times)
        summary_rows.append({
            "press": press_name, "compression_ratio": ratio,
            "context_length": int(data_dir), "avg_score": round(avg_score, 2),
            "mean_time": round(mean_time, 3),
        })

        # Collect predictions for saving
        df_preds = df_eval[["task", "answer", "predicted_answer"]].copy()
        df_preds["framework"] = "kvpress"
        df_preds["press"] = press_name
        df_preds["compression_ratio"] = ratio
        df_preds["context_length"] = int(data_dir)
        df_preds["elapsed_sec"] = [round(t, 3) for t in elapsed_times]
        all_predictions.append(df_preds)

        torch.cuda.empty_cache()

print(f"\nTotal configurations: {len(summary_rows)}")


Running: no_press | ratio=0.0 | ctx=4096 (65 examples)

Running: no_press | ratio=0.0 | ctx=4096 (65 examples)
  6/65 — 18s elapsed
  6/65 — 18s elapsed


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  12/65 — 32s elapsed
  12/65 — 32s elapsed
  18/65 — 43s elapsed
  18/65 — 43s elapsed
  24/65 — 52s elapsed
  24/65 — 52s elapsed
  30/65 — 59s elapsed
  30/65 — 59s elapsed
  36/65 — 71s elapsed
  36/65 — 71s elapsed
  42/65 — 79s elapsed
  42/65 — 79s elapsed
  48/65 — 91s elapsed
  48/65 — 91s elapsed
  54/65 — 99s elapsed
  54/65 — 99s elapsed
  60/65 — 115s elapsed
  60/65 — 115s elapsed
  65/65 — 124s elapsed
  65/65 — 124s elapsed
  Done: 124s, peak_mem=17.71GB
  Done: 124s, peak_mem=17.71GB

Running: no_press | ratio=0.0 | ctx=8192 (65 examples)

Running: no_press | ratio=0.0 | ctx=8192 (65 examples)
  6/65 — 18s elapsed
  6/65 — 18s elapsed
  12/65 — 34s elapsed
  12/65 — 34s elapsed
  18/65 — 48s elapsed
  18/65 — 48s elapsed
  24/65 — 58s elapsed
  24/65 — 58s elapsed
  30/65 — 66s elapsed
  30/65 — 66s elapsed
  36/65 — 79s elapsed
  36/65 — 79s elapsed
  42/65 — 90s elapsed
  42/65 — 90s elapsed
  48/65 — 104s elapsed
  48/65 — 104s elapsed
  54/65 — 115s elapsed
  54/65

## 4. Results

Scores computed using `calculate_metrics` from the kvpress evaluation
framework — same string-match scoring as the kvpress leaderboard.

In [7]:
summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))

           press  compression_ratio  context_length  avg_score  mean_time
        no_press               0.00            4096      93.82      1.908
        no_press               0.00            8192      94.54      2.196
full_replacement               0.01            4096      91.88      2.185
full_replacement               0.01            8192      95.58      2.488
full_replacement               0.25            4096      85.62      2.137
full_replacement               0.25            8192      80.39      2.822
full_replacement               0.50            4096      73.96      2.182
full_replacement               0.50            8192      72.06      2.763
full_replacement               0.75            4096      55.62      2.402
full_replacement               0.75            8192      62.65      2.855
       filtering               0.01            4096      91.88      5.935
       filtering               0.01            8192      95.58      6.018
       filtering               0.25   

In [8]:
for key, metrics in sorted(all_metrics.items()):
    print(f"\n{key}")
    for task, scores in sorted(metrics.items()):
        print(f"  {task:30s}: {scores['string_match']:.2f}")


filtering__0.01__4096
  cwe                           : 96.67
  fwe                           : 83.33

filtering__0.01__4096
  cwe                           : 96.67
  fwe                           : 83.33
  niah_multikey_1               : 100.00
  niah_multikey_1               : 100.00  niah_multikey_2               : 100.00

  niah_multikey_2               : 100.00
  niah_multikey_3               : 100.00
  niah_multiquery               : 100.00
  niah_multivalue               : 100.00
  niah_single_1                 : 100.00
  niah_single_2                 : 100.00
  niah_single_3                 : 100.00
  qa_1                          : 60.00
  qa_2                          : 62.50

filtering__0.01__8192
  cwe                           : 90.00
  fwe                           : 94.44
  niah_multikey_1               : 100.00
  niah_multikey_2               : 100.00
  niah_multikey_3               : 100.00
  niah_multiquery               : 100.00
  niah_multivalue               : 100

## 5. Save Results

In [9]:
import json

os.makedirs("results/kvpress_ruler", exist_ok=True)

predictions_path = "results/kvpress_ruler/predictions.csv"
df_all = pd.concat(all_predictions, ignore_index=True)
df_all.to_csv(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

metrics_path = "results/kvpress_ruler/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"Saved metrics to {metrics_path}")

Saved predictions to results/kvpress_ruler/predictions.csv
Saved predictions to results/kvpress_ruler/predictions.csv
Saved metrics to results/kvpress_ruler/metrics.json
Saved metrics to results/kvpress_ruler/metrics.json
